# Inicio

In [2]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.optim as optim
from torchmetrics.functional.classification import multiclass_f1_score
from sklearn.metrics import f1_score

import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import copy
from copy import deepcopy
from tqdm import tqdm
import time
import os
from scipy.spatial import distance
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [3]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [4]:
def recortar_janelas(acc, J, passo):
    N = acc.shape[0]
    Nj = (N - J) // passo + 1
    janelas = np.zeros((Nj, J, 3))
    for i in range(Nj):
        janelas[i] = acc[i * passo:i * passo + J]
    return janelas

In [5]:
actis = ['climbingdown', 'climbingup', 'jumping', 'lying', 'running', 'sitting', 'standing', 'walking']
posis = ['chest', 'forearm', 'head', 'shin', 'thigh', 'upperarm', 'waist']
users = ['proband' + x for x in np.arange(1,16).astype(str)]

In [ ]:
data = np.load('/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/notebooks/Xydata.npz')
Xdata = data['Xdata']
ydata = data['ydata']

In [8]:
# pasta = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/Dataset/realworldcsvs/'
# J = 150
# Xdata = []
# ydata = []
# for user in tqdm(users):
#     for i, pos in enumerate(posis):
#         for j, act in enumerate(actis):
#             files = os.listdir(pasta+user+'/acc/')
#             inds = [(file.find(act)>-1) and (file.find(pos)>-1) for file in files]
#             if np.array(inds).any():
#                 ind = inds.index(True)
#                 acc = pd.read_csv(pasta+user+'/acc/'+files[ind]).values[:,2:]
#                 janelas = recortar_janelas(acc, J, J)
#                 rotulos = np.full((janelas.shape[0], 2), [i, j], dtype=int)
#                 Xdata.append(janelas)
#                 ydata.append(rotulos)
# Xdata = np.concatenate(Xdata, axis=0)
# Xdata = Xdata/20
# ydata = np.concatenate(ydata, axis=0)
# Xdata.shape, ydata.shape

100%|██████████| 15/15 [13:43<00:00, 54.87s/it]


((151481, 150, 3), (151481, 2))

# Modelo chang

In [10]:
class ChangEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(in_channels=3, out_channels=16, kernel_size=3)
        self.inst1 = nn.InstanceNorm1d(16, affine=True)
        self.drop1 = nn.Dropout(p=0.2)

        self.conv2 = nn.Conv1d(in_channels=16, out_channels=16, kernel_size=3)
        self.inst2 = nn.InstanceNorm1d(16, affine=True)
        self.drop2 = nn.Dropout(p=0.2)

        self.conv3 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, stride=4)
        self.inst3 = nn.InstanceNorm1d(32, affine=True)
        self.drop3 = nn.Dropout(p=0.2)

        self.conv4 = nn.Conv1d(in_channels=32, out_channels=32, kernel_size=3, stride=1)
        self.inst4 = nn.InstanceNorm1d(32, affine=True)
        self.drop4 = nn.Dropout(p=0.2)

        self.conv5 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, stride=4)
        self.inst5 = nn.InstanceNorm1d(64, affine=True)
        self.drop5 = nn.Dropout(p=0.2)

        self.conv6 = nn.Conv1d(in_channels=64, out_channels=100, kernel_size=5, stride=1)

        self.relu = nn.LeakyReLU(0.3)
        self.glap = nn.AvgPool1d(kernel_size=4)

    def forward(self, x):
        # (N,T,C) -> (N,C,T)
        x = x.transpose(1, 2)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.inst1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.inst2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.relu(x)
        x = self.inst3(x)
        x = self.drop3(x)

        x = self.conv4(x)
        x = self.relu(x)
        x = self.inst4(x)
        x = self.drop4(x)

        x = self.conv5(x)
        x = self.relu(x)
        x = self.inst5(x)
        x = self.drop5(x)

        x = self.conv6(x)
        x = self.relu(x)

        x = self.glap(x)
        x = x.flatten(start_dim=1)

        logits = x
        return logits

In [11]:
class ChangClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.densa = nn.Linear(in_features=100, out_features=8)

    def forward(self, x):
        logits = self.densa(x)
        return logits

# Treinamento

In [12]:
inds = ydata[:,0]==0
X = Xdata[inds]
y = ydata[inds][:,1]
X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=1, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.1, random_state=1, stratify=y_train)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=125, shuffle=True, pin_memory=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False, pin_memory=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)

In [13]:
@torch.no_grad()
def evaluate(encoder, classifier, loader, loss_fn, device):
    encoder.eval()
    classifier.eval()
    total_loss = 0
    total_samples = 0
    y_true = []
    y_pred = []

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = classifier(encoder(X))
        loss = loss_fn(logits, y)
        total_loss += loss.item() * len(y)
        total_samples += len(y)
        pred = torch.argmax(logits, dim=1)
        y_true.append(y)
        y_pred.append(pred)

    y_true = torch.cat(y_true)
    y_pred = torch.cat(y_pred)
    f1 = multiclass_f1_score(y_pred, y_true, num_classes=8, average="macro").item()

    return total_loss / total_samples, f1

In [14]:
def train_model(train_loader, val_loader, device):
    encoder = ChangEncoder().to(device)
    classifier = ChangClassifier().to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(list(encoder.parameters()) + list(classifier.parameters()), lr=1e-3)
    n_epochs = 100
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_f1": [],
        "val_f1": []
    }
    best_f1 = -1
    best_encoder = None
    best_classifier = None

    for epoch in range(n_epochs):
        encoder.train()
        classifier.train()
        running_loss = 0
        n_samples = 0
        bar = tqdm(train_loader)

        for X, y in bar:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad()
            logits = classifier(encoder(X))
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(y)
            n_samples += len(y)
            bar.set_description(f"Epoch {epoch+1}")
            bar.set_postfix(loss=loss.item())

        train_loss, train_f1 = evaluate(
            encoder,
            classifier,
            train_loader,
            loss_fn,
            device
        )

        val_loss, val_f1 = evaluate(
            encoder,
            classifier,
            val_loader,
            loss_fn,
            device
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)

        print(
            f"Epoch {epoch+1:3d} | "
            f"Train F1={train_f1:.4f} | "
            f"Val F1={val_f1:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_encoder = deepcopy(encoder)
            best_classifier = deepcopy(classifier)

    return best_encoder, best_classifier, history

In [15]:
enc, cla, history = train_model(train_loader, val_loader, device)

Epoch 1: 100%|██████████| 125/125 [00:02<00:00, 53.96it/s, loss=0.92] 


Epoch   1 | Train F1=0.5345 | Val F1=0.5308


Epoch 2: 100%|██████████| 125/125 [00:01<00:00, 108.21it/s, loss=0.881]


Epoch   2 | Train F1=0.6256 | Val F1=0.6122


Epoch 3: 100%|██████████| 125/125 [00:01<00:00, 110.72it/s, loss=0.735]


...

Epoch  98 | Train F1=0.9506 | Val F1=0.8904


Epoch 99: 100%|██████████| 125/125 [00:01<00:00, 85.95it/s, loss=0.267]


Epoch  99 | Train F1=0.9435 | Val F1=0.8943


Epoch 100: 100%|██████████| 125/125 [00:01<00:00, 80.72it/s, loss=0.265]


Epoch 100 | Train F1=0.9489 | Val F1=0.9086


In [16]:
test_loss, test_f1 = evaluate(
    enc,
    cla,
    test_loader,
    nn.CrossEntropyLoss(),
    device
)

print(test_loss)
print(test_f1)

0.39336768785425935
0.9001853466033936


In [19]:
px.line(history["train_loss"], title="Loss")
px.line(history["val_f1"], title="Loss")